# Azure AI Search Demo with Hotels Data

This notebook demonstrates three types of search capabilities using Azure AI Search:

1. **Keyword Search** - Traditional full-text search with exact term matching
2. **Semantic Search** - AI-powered search that understands natural language and intent
3. **Hybrid Search** - Combination of multiple search methods for best results

## Prerequisites

Create a `.env` file in the same directory with:
```
AZURE_SEARCH_ENDPOINT=https://your-search-service.search.windows.net
AZURE_SEARCH_KEY=your-admin-key-here
AZURE_SEARCH_INDEX=hotels
```

In [ ]:
# Load diagram rendering helper
from IPython.display import HTML, display
import base64

def render_mermaid(diagram_code):
    """Render Mermaid diagrams as images"""
    graphbytes = diagram_code.encode("utf8")
    base64_bytes = base64.b64encode(graphbytes)
    base64_string = base64_bytes.decode("ascii")
    img_url = f"https://mermaid.ink/img/{base64_string}"
    html = f'<img src="{img_url}" alt="Diagram" style="max-width: 100%; height: auto;"/>'
    display(HTML(html))

print("✅ Diagram helper loaded - ready to render visualizations!")

In [ ]:
# Install required packages
%pip install -q azure-search-documents python-dotenv azure-identity
print("✅ Packages installed")

## 🚀 Quick Start

**Run these cells in order:**
1. **Cell 2** → Install packages
2. **Cell 3** → Load diagram helper
3. **Run All** → Execute complete notebook with diagrams and examples

In [5]:
# What You'll Learn - Visual Overview
diagram = """
mindmap
  root((Azure AI<br/>Search Demo))
    Keyword Search
      Exact matching
      BM25 algorithm
      Filters
      Field-specific
    Semantic Search
      Natural language
      AI re-ranking
      Captions
      Answers
    Hybrid Search
      Keyword + Semantic
      Vector + Keyword
      Full hybrid
      Best quality
    Advanced Features
      Faceting
      Suggestions
      Scoring profiles
      Optimization
"""
render_mermaid(diagram)

## What You'll Learn

Visual overview of all search concepts covered in this notebook:

## 1. Setup and Configuration

Load environment variables and initialize the Azure Search client.

In [4]:
# Architecture Overview Diagram
diagram = """
graph TB
    User["User"] --> Query["Search Query"]
    Query --> SearchService["Azure AI Search Service"]
    
    SearchService --> K["Keyword Search<br/>BM25 Algorithm"]
    SearchService --> S["Semantic Search<br/>L2 Re-ranker"]
    SearchService --> V["Vector Search<br/>Cosine Similarity"]
    
    K --> Index["Hotels Index"]
    S --> Index
    V --> Index
    
    Index --> Results["Search Results"]
    Results --> User
    
    style SearchService fill:#0078D4,color:#fff
    style K fill:#2196F3,color:#fff
    style S fill:#9C27B0,color:#fff
    style V fill:#4CAF50,color:#fff
    style Results fill:#FF9800,color:#fff
"""
render_mermaid(diagram)

### Architecture Overview

How Azure AI Search processes queries across three search engines:

In [3]:
import os
from dotenv import load_dotenv
from azure.search.documents import SearchClient
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.models import VectorizedQuery
import json

# Load environment variables
load_dotenv()

# Azure Search configuration
search_endpoint = os.getenv("AZURE_SEARCH_ENDPOINT")
search_key = os.getenv("AZURE_SEARCH_KEY")
index_name = os.getenv("AZURE_SEARCH_INDEX", "hotels")

# Validate configuration
if not search_endpoint or not search_key:
    raise ValueError("Please set AZURE_SEARCH_ENDPOINT and AZURE_SEARCH_KEY in your .env file")

# Create search client
search_client = SearchClient(
    endpoint=search_endpoint,
    index_name=index_name,
    credential=AzureKeyCredential(search_key)
)

print(f"✅ Connected to Azure Search")
print(f"   Endpoint: {search_endpoint}")
print(f"   Index: {index_name}")

ValueError: Please set AZURE_SEARCH_ENDPOINT and AZURE_SEARCH_KEY in your .env file

## 2. Helper Functions

Utility functions to display search results in a readable format.

In [9]:
# Connection Flow Diagram
diagram = """
sequenceDiagram
    participant NB as Notebook
    participant ENV as .env File
    participant SDK as Azure SDK
    participant AS as Azure Search Service
    
    NB->>ENV: Load credentials
    ENV-->>NB: Endpoint, Key, Index
    NB->>SDK: Create SearchClient
    SDK->>AS: Authenticate
    AS-->>SDK: Connected
    SDK-->>NB: Client ready
    
    Note over NB,AS: Ready to execute searches
"""
render_mermaid(diagram)

### Connection Flow

Understanding how the notebook connects to your Azure Search service:

In [ ]:
def print_search_results(results, search_type="Search", show_score=True, show_captions=False):
    """Display search results in a clean format"""
    print(f"\n{'='*80}")
    print(f"{search_type} Results")
    print(f"{'='*80}\n")
    
    count = 0
    for result in results:
        count += 1
        
        # Extract basic info
        hotel_name = result.get('HotelName', 'N/A')
        description = result.get('Description', 'N/A')
        category = result.get('Category', 'N/A')
        tags = result.get('Tags', [])
        
        print(f"Result #{count}")
        print(f"-" * 80)
        print(f"🏨 Hotel: {hotel_name}")
        print(f"📝 Category: {category}")
        
        if show_score and hasattr(result, '@search.score'):
            print(f"⭐ Score: {result['@search.score']:.4f}")
        
        if show_captions and hasattr(result, '@search.captions'):
            captions = result.get('@search.captions', [])
            if captions:
                print(f"\n💡 Semantic Caption:")
                for caption in captions:
                    print(f"   {caption.text}")
        
        # Truncate description
        desc_preview = description[:200] + "..." if len(description) > 200 else description
        print(f"\n📄 Description: {desc_preview}")
        
        if tags:
            print(f"🏷️  Tags: {', '.join(tags[:5])}")
        
        print(f"\n")
    
    if count == 0:
        print("No results found.")
    else:
        print(f"Total results displayed: {count}")
    print(f"{'='*80}\n")


def print_query_info(query_text, query_type):
    """Display query information"""
    print(f"\n{'🔍 ' + query_type}")
    print(f"Query: \"{query_text}\"")

In [8]:
# Keyword Search Execution Flow
diagram = """
sequenceDiagram
    participant User as You (Notebook)
    participant SDK as Azure SDK
    participant Engine as Search Engine
    participant Index as Hotels Index
    
    User->>SDK: search("wifi pool")
    SDK->>Engine: Process query
    
    Engine->>Engine: Tokenize: ["wifi", "pool"]
    Engine->>Engine: Normalize and stem terms
    
    Engine->>Index: Look up "wifi" in inverted index
    Index-->>Engine: Hotels: [1, 5, 12, 23]
    
    Engine->>Index: Look up "pool" in inverted index
    Index-->>Engine: Hotels: [5, 12, 18, 23]
    
    Engine->>Engine: Calculate BM25 scores
    
    Engine->>Index: Fetch fields for top results
    Index-->>Engine: HotelName, Description, Category, Tags
    
    Engine-->>SDK: Ranked results with scores
    SDK-->>User: Display formatted results
    
    Note over User,Index: Fast! Typically < 100ms
"""
render_mermaid(diagram)

### Visual: Keyword Search Execution Flow

Step-by-step execution of a keyword search query:

## 3. Keyword Search (Full-Text Search)

Traditional search that matches exact words in your query against indexed fields.

**How it works:**
- Tokenizes query into individual terms
- Searches inverted index for exact matches
- Uses BM25 ranking algorithm
- Best for: Exact terms, product IDs, categories, specific phrases

**Example:** Searching for hotels with specific amenities or categories.

### Visual: Keyword Search Process

How keyword search tokenizes and matches terms:

In [11]:
# Keyword Search Process Diagram
diagram = """
flowchart LR
    Query["Query: 'wifi pool'"] --> Tokenize["Tokenize"]
    Tokenize --> T1["wifi"]
    Tokenize --> T2["pool"]
    
    T1 --> InvertedIndex["Inverted Index"]
    T2 --> InvertedIndex
    
    InvertedIndex --> Match["Match Documents"]
    Match --> BM25["BM25 Scoring"]
    BM25 --> Rank["Ranked Results"]
    
    style Query fill:#E3F2FD
    style InvertedIndex fill:#2196F3,color:#fff
    style Rank fill:#4CAF50,color:#fff
"""
render_mermaid(diagram)

In [ ]:
# Example 1: Search for "wifi pool" - exact term matching
query = "wifi pool"
print_query_info(query, "Keyword Search - Exact Terms")

results = search_client.search(
    search_text=query,
    select=["HotelName", "Description", "Category", "Tags"],
    top=5
)

print_search_results(results, "Keyword Search", show_score=True)

In [ ]:
# Example 2: Search with filters
query = "hotel"
print_query_info(query, "Keyword Search - With Filters")

results = search_client.search(
    search_text=query,
    filter="Category eq 'Boutique'",
    select=["HotelName", "Description", "Category", "Tags"],
    top=5
)

print_search_results(results, "Keyword Search + Filter (Category=Boutique)", show_score=True)

In [13]:
# Semantic Search with Hotels Data
diagram = """
graph TB
    Query["Query: 'cozy place near tourist attractions'"]
    
    Query --> Phase1["Phase 1: Keyword Search"]
    Phase1 --> KW["Search for: cozy, place, tourist, attractions"]
    KW --> Initial["Initial Results: 50 hotels with some keywords"]
    
    Initial --> Phase2["Phase 2: Semantic Understanding"]
    Phase2 --> L2Model["Microsoft L2 Model analyzes"]
    L2Model --> Concept1["'cozy' means comfortable, intimate, welcoming"]
    L2Model --> Concept2["'tourist attractions' means landmarks, sightseeing"]
    
    Concept1 --> Match["Find hotels matching concepts: Stay-Kay City Hotel"]
    Concept2 --> Match
    
    Match --> Phase3["Phase 3: Generate Insights"]
    Phase3 --> Caption["Caption: 'Classic Hotel with fully-refurbished rooms near Times Square...'"]
    Phase3 --> Score["Relevance Score: 0.87"]
    
    Caption --> Results["Final Results with Context"]
    Score --> Results
    
    style Phase1 fill:#E3F2FD
    style Phase2 fill:#9C27B0,color:#fff
    style Phase3 fill:#4CAF50,color:#fff
    style Results fill:#FF9800,color:#fff
"""
render_mermaid(diagram)

### Visual: Semantic Search with Real Hotel Data

How semantic search understands "cozy place near tourist attractions":

In [ ]:
# Example 3: Search specific fields with weights
query = "refurbished"
print_query_info(query, "Keyword Search - Field-Specific")

results = search_client.search(
    search_text=query,
    search_fields=["Description", "Tags"],  # Only search in these fields
    select=["HotelName", "Description", "Category", "Tags"],
    top=5
)

print_search_results(results, "Keyword Search (Description & Tags only)", show_score=True)

## 4. Semantic Search

AI-powered search using Microsoft's semantic ranker. Understands natural language queries and user intent.

**How it works:**
- First performs keyword search to get top 50 results
- Applies Microsoft L2 deep learning model to re-rank results
- Understands synonyms, context, and semantic meaning
- Generates captions highlighting relevant content
- Can provide direct answers to questions

**Best for:** Natural language questions, concept-based queries, "find similar" requests

**Note:** Semantic search requires:
- Azure Search Premium tier
- Semantic configuration in your index
- `queryType="semantic"` parameter

### Visual: Semantic Search Architecture

Three-phase process of semantic search:

In [14]:
# Semantic Search Architecture
diagram = """
flowchart TB
    Query["Natural Language Query: 'cozy place near attractions'"]
    
    Query --> Step1["Step 1: Keyword Search"]
    Step1 --> Top50["Top 50 Results"]
    
    Top50 --> Step2["Step 2: L2 Semantic Model"]
    Step2 --> Understanding["Understand Intent: cozy = comfortable, attractions = tourist sites"]
    
    Understanding --> Step3["Step 3: Re-rank Results"]
    Step3 --> Captions["Generate Captions"]
    Step3 --> Answers["Extract Answers"]
    
    Captions --> Final["Final Results with Context"]
    Answers --> Final
    
    style Query fill:#E1BEE7
    style Step2 fill:#9C27B0,color:#fff
    style Final fill:#4CAF50,color:#fff
"""
render_mermaid(diagram)

In [ ]:
# Example 1: Natural language query
query = "cozy place to stay near tourist attractions"
print_query_info(query, "Semantic Search - Natural Language")

results = search_client.search(
    search_text=query,
    query_type="semantic",
    semantic_configuration_name="default",  # Update with your semantic config name
    select=["HotelName", "Description", "Category", "Tags"],
    top=5
)

print_search_results(results, "Semantic Search", show_score=True, show_captions=True)

In [ ]:
# Example 2: Question-based query
query = "Which hotels are good for business travelers?"
print_query_info(query, "Semantic Search - Question Format")

results = search_client.search(
    search_text=query,
    query_type="semantic",
    semantic_configuration_name="default",
    query_answer="extractive",  # Request direct answers
    select=["HotelName", "Description", "Category", "Tags"],
    top=5
)

# Check for semantic answers
print("\n" + "="*80)
print("Semantic Answers (Direct Responses)")
print("="*80 + "\n")

if hasattr(results, 'get_answers') and results.get_answers():
    for answer in results.get_answers():
        print(f"💬 Answer: {answer.text}")
        print(f"   Score: {answer.score:.4f}\n")
else:
    print("No direct answers generated for this query.\n")

print_search_results(results, "Semantic Search (Question)", show_score=True, show_captions=True)

In [ ]:
# Example 3: Concept-based search
query = "romantic getaway with scenic views"
print_query_info(query, "Semantic Search - Concept-Based")

results = search_client.search(
    search_text=query,
    query_type="semantic",
    semantic_configuration_name="default",
    select=["HotelName", "Description", "Category", "Tags"],
    top=5
)

print_search_results(results, "Semantic Search (Concept)", show_score=True, show_captions=True)

## 5. Hybrid Search

Combines multiple search techniques for superior results. There are several hybrid approaches:

### 5.1 Semantic + Keyword (Most Common)
- Performs keyword search first
- Applies semantic re-ranking on top results
- Best balance of precision and understanding

### 5.2 Vector + Keyword (Requires embeddings)
- Combines text matching with semantic similarity
- Uses Reciprocal Rank Fusion (RRF) to merge results
- Requires vector embeddings in your index

### 5.3 Full Hybrid (All three)
- Keyword + Vector (RRF merged) + Semantic re-ranking
- Maximum search quality
- Higher cost and complexity

### Visual: Hybrid Search Comparison

Three hybrid approaches and their benefits:

In [12]:
# Hybrid Search Comparison Diagram
diagram = """
graph TB
    subgraph KeywordSemantic["Keyword + Semantic"]
        KS1["Keyword Search"] --> KS2["Top 50 Results"]
        KS2 --> KS3["Semantic Re-rank"]
        KS3 --> KS4["Good Balance"]
    end
    
    subgraph VectorKeyword["Vector + Keyword"]
        VK1["Keyword Match"] --> VK2["RRF Merge"]
        VK3["Vector Similarity"] --> VK2
        VK2 --> VK4["Best Similarity"]
    end
    
    subgraph FullHybrid["Full Hybrid"]
        FH1["Keyword"] --> FH2["RRF"]
        FH3["Vector"] --> FH2
        FH2 --> FH4["Top Results"]
        FH4 --> FH5["Semantic Re-rank"]
        FH5 --> FH6["Maximum Quality"]
    end
    
    style KS4 fill:#2196F3,color:#fff
    style VK4 fill:#4CAF50,color:#fff
    style FH6 fill:#FF9800,color:#fff
"""
render_mermaid(diagram)

### 5.1 Hybrid: Keyword + Semantic

This is the easiest and most common hybrid approach. It automatically happens when you use `queryType="semantic"`.

In [15]:
# How Hybrid (Keyword + Semantic) Works
diagram = """
sequenceDiagram
    participant User
    participant Keyword as Keyword Engine
    participant Semantic as Semantic Ranker
    participant Index as Hotels Index
    
    User->>Keyword: Query: "affordable modern hotel"
    Keyword->>Index: Search for keywords
    Index-->>Keyword: 50 matching hotels
    
    Keyword->>Semantic: Pass top 50 results
    Semantic->>Semantic: Apply L2 model - Understand context
    Semantic->>Semantic: Re-rank by relevance
    Semantic->>Semantic: Generate captions
    
    Semantic-->>User: Top 5 most relevant with explanations
    
    Note over Keyword,Semantic: Best of both worlds!
"""
render_mermaid(diagram)

### Visual: Hybrid Search Workflow

How keyword and semantic search work together:

In [ ]:
# Hybrid Search: Keyword foundation + Semantic re-ranking
query = "affordable hotel with modern amenities and good location"
print_query_info(query, "Hybrid Search (Keyword + Semantic)")

results = search_client.search(
    search_text=query,
    query_type="semantic",
    semantic_configuration_name="default",
    
    # Keyword parameters
    search_fields=["HotelName", "Description", "Tags"],
    filter=None,  # Add filters if needed
    
    # Semantic parameters
    query_caption="extractive",
    query_answer="extractive",
    
    select=["HotelName", "Description", "Category", "Tags", "ParkingIncluded", "Rating"],
    top=5
)

print_search_results(results, "Hybrid: Keyword + Semantic", show_score=True, show_captions=True)

In [ ]:
# Hybrid with scoring profile (boost certain fields)
query = "hotel with meeting rooms"
print_query_info(query, "Hybrid Search with Scoring Profile")

results = search_client.search(
    search_text=query,
    query_type="semantic",
    semantic_configuration_name="default",
    
    # Apply custom scoring profile (if configured in your index)
    # scoring_profile="business-traveler-boost",
    
    select=["HotelName", "Description", "Category", "Tags"],
    top=5
)

print_search_results(results, "Hybrid with Custom Scoring", show_score=True, show_captions=True)

### 5.2 Hybrid: Vector + Keyword (Advanced)

**Note:** This requires:
1. Vector fields in your index (e.g., `descriptionVector`)
2. Embedding generation (using OpenAI or other models)
3. Vector search configuration

If your index has vector fields, uncomment and run the code below.

In [ ]:
# EXAMPLE: Hybrid Vector + Keyword Search
# Uncomment and modify if you have vectors in your index

"""
from openai import AzureOpenAI

# Initialize OpenAI client for embeddings
openai_client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_KEY"),
    api_version="2024-02-01",
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT")
)

# Generate embedding for query
query = "luxury hotel with spa and ocean view"
embedding_response = openai_client.embeddings.create(
    input=query,
    model="text-embedding-ada-002"
)
query_vector = embedding_response.data[0].embedding

# Hybrid search: Keyword + Vector
print_query_info(query, "Hybrid Search (Keyword + Vector)")

vector_query = VectorizedQuery(
    vector=query_vector,
    k_nearest_neighbors=50,
    fields="descriptionVector"
)

results = search_client.search(
    search_text=query,  # Keyword component
    vector_queries=[vector_query],  # Vector component
    select=["HotelName", "Description", "Category", "Tags"],
    top=5
)

print_search_results(results, "Hybrid: Keyword + Vector", show_score=True)
"""

print("⚠️  Vector + Keyword hybrid search requires:")
print("   1. Vector fields in your index")
print("   2. OpenAI or embedding service configured")
print("   3. Uncomment and run the code above after setup")

### 5.3 Full Hybrid: Keyword + Vector + Semantic

The ultimate search quality - combines all three methods!

In [ ]:
# EXAMPLE: Full Hybrid Search (all three methods)
# Uncomment if you have vector embeddings configured

"""
# Generate embedding
query = "family-friendly resort with kids activities"
embedding_response = openai_client.embeddings.create(
    input=query,
    model="text-embedding-ada-002"
)
query_vector = embedding_response.data[0].embedding

# Full hybrid: Keyword + Vector + Semantic
print_query_info(query, "Full Hybrid Search (Keyword + Vector + Semantic)")

vector_query = VectorizedQuery(
    vector=query_vector,
    k_nearest_neighbors=50,
    fields="descriptionVector"
)

results = search_client.search(
    search_text=query,                    # Keyword search
    vector_queries=[vector_query],        # Vector search
    query_type="semantic",                # Semantic re-ranking
    semantic_configuration_name="default",
    query_caption="extractive",
    
    select=["HotelName", "Description", "Category", "Tags"],
    top=5
)

print_search_results(results, "Full Hybrid (All Three Methods)", show_score=True, show_captions=True)
"""

print("⚠️  Full hybrid search combines all three methods:")
print("   • Keyword (text matching)")
print("   • Vector (semantic similarity)")  
print("   • Semantic (AI re-ranking)")
print("\n   Requires vector fields + semantic configuration")
print("   Uncomment and run the code above after setup")

## 6. Comparison: All Search Types

Run different search types on the same query to see the differences.

### Visual: Three Methods Side-by-Side

Comparing how each method handles the same query:

In [17]:
# Visual Comparison: Three Search Methods
diagram = """
graph TB
    Query["User Query: 'hotel with great location and comfortable rooms'"]
    
    subgraph KeywordSearch["Keyword Search"]
        KW1[Tokenize query] --> KW2[Search inverted index]
        KW2 --> KW3["Match exact words: hotel, location, comfortable, rooms"]
        KW3 --> KW4["BM25 Scoring"]
        KW4 --> KW5["Results ranked by term frequency"]
    end
    
    subgraph SemanticSearch["Semantic Search"]
        SS1[Keyword search first] --> SS2[Get top 50 results]
        SS2 --> SS3[L2 Semantic Model]
        SS3 --> SS4["Understand concepts: location = centrality, comfortable = quality"]
        SS4 --> SS5["Re-rank by relevance - Generate captions"]
    end
    
    subgraph HybridSearch["Hybrid Search"]
        HS1[Keyword foundation] --> HS2[Top 50 matches]
        HS2 --> HS3[Semantic re-ranking]
        HS3 --> HS4["Best of both: Precision + Understanding"]
    end
    
    Query --> KeywordSearch
    Query --> SemanticSearch
    Query --> HybridSearch
    
    style KW5 fill:#2196F3,color:#fff
    style SS5 fill:#9C27B0,color:#fff
    style HS4 fill:#FF9800,color:#fff
"""
render_mermaid(diagram)

In [ ]:
# Compare all search types with the same query
comparison_query = "hotel with great location and comfortable rooms"

print("\n" + "="*80)
print(f"COMPARING SEARCH TYPES")
print(f"Query: \"{comparison_query}\"")
print("="*80)

# 1. Keyword Search
print("\n\n1️⃣  KEYWORD SEARCH")
print("-" * 80)
keyword_results = search_client.search(
    search_text=comparison_query,
    select=["HotelName", "Description", "Category"],
    top=3
)
for i, result in enumerate(keyword_results, 1):
    print(f"\n{i}. {result['HotelName']}")
    print(f"   Score: {result['@search.score']:.4f}")
    print(f"   Category: {result.get('Category', 'N/A')}")

# 2. Semantic Search
print("\n\n2️⃣  SEMANTIC SEARCH")
print("-" * 80)
semantic_results = search_client.search(
    search_text=comparison_query,
    query_type="semantic",
    semantic_configuration_name="default",
    select=["HotelName", "Description", "Category"],
    top=3
)
for i, result in enumerate(semantic_results, 1):
    print(f"\n{i}. {result['HotelName']}")
    print(f"   Score: {result['@search.score']:.4f}")
    print(f"   Category: {result.get('Category', 'N/A')}")
    
    # Show captions if available
    if hasattr(result, '@search.captions'):
        captions = result.get('@search.captions', [])
        if captions:
            print(f"   Caption: {captions[0].text[:100]}...")

# 3. Hybrid (Keyword + Semantic)
print("\n\n3️⃣  HYBRID SEARCH (Keyword + Semantic)")
print("-" * 80)
hybrid_results = search_client.search(
    search_text=comparison_query,
    query_type="semantic",
    semantic_configuration_name="default",
    query_caption="extractive",
    select=["HotelName", "Description", "Category"],
    top=3
)
for i, result in enumerate(hybrid_results, 1):
    print(f"\n{i}. {result['HotelName']}")
    print(f"   Score: {result['@search.score']:.4f}")
    print(f"   Category: {result.get('Category', 'N/A')}")

print("\n" + "="*80 + "\n")

## 7. Advanced Examples

More sophisticated queries demonstrating advanced features.

### Visual: Advanced Features

Architecture of faceting, suggestions, and scoring:

In [18]:
# Advanced Features Architecture
diagram = """
graph LR
    User["User Input"] --> Features["Advanced Features"]
    
    Features --> Facets["Faceting - Category aggregations"]
    Features --> Suggest["Suggestions - Type-ahead"]
    Features --> Auto["Autocomplete - Query completion"]
    Features --> Scoring["Scoring Profiles - Custom ranking"]
    
    Facets --> UI1["Filter UI"]
    Suggest --> UI2["Search Box"]
    Auto --> UI2
    Scoring --> UI3["Sorted Results"]
    
    Index["Hotels Index"] --> Facets
    Index --> Suggest
    Index --> Auto
    Index --> Scoring
    
    style Features fill:#0078D4,color:#fff
    style Index fill:#4CAF50,color:#fff
"""
render_mermaid(diagram)

In [ ]:
# Example: Faceted search (aggregations)
query = "*"  # Get all documents
print_query_info(query, "Faceted Search - Aggregations")

results = search_client.search(
    search_text=query,
    facets=["Category", "Tags,count:10", "ParkingIncluded"],
    select=["HotelName"],
    top=1  # We only want facets, not results
)

print("\n" + "="*80)
print("Faceted Search Results (Aggregations)")
print("="*80 + "\n")

# Display facets
if hasattr(results, 'get_facets'):
    facets = results.get_facets()
    
    for facet_name, facet_values in facets.items():
        print(f"\n📊 {facet_name}:")
        print("-" * 40)
        for value in facet_values:
            print(f"   • {value.get('value', 'N/A')}: {value.get('count', 0)} hotels")

print("\n" + "="*80 + "\n")

In [ ]:
# Example: Autocomplete/Suggestions
query_prefix = "lux"
print(f"\n🔍 Autocomplete Suggestions")
print(f"Prefix: \"{query_prefix}\"")
print("="*80 + "\n")

# Note: Requires suggester configured in index
try:
    suggestions = search_client.suggest(
        search_text=query_prefix,
        suggester_name="sg",  # Update with your suggester name
        select=["HotelName", "Category"],
        top=5
    )
    
    print("Suggestions:")
    for i, suggestion in enumerate(suggestions, 1):
        print(f"{i}. {suggestion['HotelName']} ({suggestion.get('Category', 'N/A')})")
        
except Exception as e:
    print(f"⚠️  Suggester not configured: {e}")
    print("To enable autocomplete, add a suggester to your index definition.")

print("\n" + "="*80 + "\n")

## 8. Search Best Practices

### When to Use Each Search Type:

| Search Type | Best For | Example Query |
|-------------|----------|---------------|
| **Keyword** | Exact terms, IDs, categories, filters | "HotelId: 12", "Category: Boutique", "wifi parking" |
| **Semantic** | Natural language, questions, concepts | "Where can I find a romantic hotel?", "cozy place near attractions" |
| **Hybrid** | Best overall quality, complex queries | "affordable luxury hotel with spa near beach" |

### Performance Tips:

1. **Use `select`** - Only retrieve fields you need
2. **Apply filters** - Pre-filter before searching
3. **Limit `top`** - Don't retrieve more results than needed  
4. **Cache common queries** - Store frequent search results
5. **Use pagination** - Implement `skip` and `top` for large result sets

### Cost Optimization:

- **Keyword**: Included in base pricing ✅
- **Semantic**: +$3 per 1,000 queries (Premium tier required) 💰
- **Vector**: Storage costs (4x-32x larger index) + embedding API costs 💰💰

### Visual: Cost Comparison

Understanding pricing across search methods:

In [19]:
# Cost Comparison Chart
diagram = """
graph TD
    Cost["Search Method Costs"] --> K[Keyword Search]
    Cost --> S[Semantic Search]
    Cost --> V[Vector Search]
    Cost --> H[Hybrid Search]
    
    K --> K1["Index: Base cost"]
    K --> K2["Query: Included"]
    K --> K3["Tier: Standard"]
    K --> KTotal["Total: LOW"]
    
    S --> S1["Index: Base cost"]
    S --> S2["Query: +3USD per 1000"]
    S --> S3["Tier: Premium required"]
    S --> STotal["Total: MEDIUM"]
    
    V --> V1["Index: 4x-32x larger"]
    V --> V2["Query: Embedding API"]
    V --> V3["Tier: Standard"]
    V --> VTotal["Total: HIGH"]
    
    H --> H1["Index: Base + vectors"]
    H --> H2["Query: Semantic + API"]
    H --> H3["Tier: Premium"]
    H --> HTotal["Total: HIGHEST"]
    
    style KTotal fill:#4CAF50,color:#fff
    style STotal fill:#FF9800,color:#fff
    style VTotal fill:#FF5722,color:#fff
    style HTotal fill:#9C27B0,color:#fff
"""
render_mermaid(diagram)

### Visual: Performance Optimization

Five-step optimization workflow:

In [20]:
# Performance Optimization Flow
diagram = """
flowchart LR
    Query["Search Query"] --> Optimize["Optimization Steps"]
    
    Optimize --> Step1["1. Use 'select' - Only needed fields"]
    Optimize --> Step2["2. Apply filters - Pre-filter results"]
    Optimize --> Step3["3. Limit 'top' - Fewer results"]
    Optimize --> Step4["4. Cache - Common queries"]
    Optimize --> Step5["5. Pagination - Skip + Top"]
    
    Step1 --> Perf1["Reduce Network traffic"]
    Step2 --> Perf2["Reduce Processing time"]
    Step3 --> Perf3["Reduce Response size"]
    Step4 --> Perf4["Reduce API calls"]
    Step5 --> Perf5["Reduce Load time"]
    
    Perf1 --> Result["Fast Search"]
    Perf2 --> Result
    Perf3 --> Result
    Perf4 --> Result
    Perf5 --> Result
    
    style Optimize fill:#0078D4,color:#fff
    style Result fill:#4CAF50,color:#fff
"""
render_mermaid(diagram)

### Visual: Decision Tree

Choose the right search type for your use case:

In [30]:
# Decision Tree: Choose Your Search Type
diagram = """
flowchart TD
    Start{What's your use case?}
    
    Start -->|Exact match needed - Product IDs, Categories| Keyword["Use Keyword Search - Fast - Precise - Low cost"]
    
    Start -->|Natural language - Questions, Concepts| CheckTier{Premium tier?}
    CheckTier -->|Yes| Semantic["Use Semantic Search - Understanding - Captions - Answers"]
    CheckTier -->|No| Keyword2["Use Keyword Search - (Semantic requires Premium)"]
    
    Start -->|Best quality - Complex queries| Hybrid["Use Hybrid Search - Maximum relevance - Best UX - Higher cost"]
    
    Start -->|Find similar items - Recommendations| Vector["Use Vector Search - Semantic similarity - ML-powered - Storage costs"]
    
    Keyword --> Optimize["+ Add filters + Use scoring profiles"]
    Semantic --> Optimize2["+ Configure semantic config + Enable captions"]
    Hybrid --> Optimize3["+ Combine with vectors + Fine-tune ranking"]
    Vector --> Optimize4["+ Add embeddings + Configure HNSW"]
    
    style Keyword fill:#2196F3,color:#fff
    style Semantic fill:#9C27B0,color:#fff
    style Hybrid fill:#FF9800,color:#fff
    style Vector fill:#4CAF50,color:#fff
"""
render_mermaid(diagram)

## 9. Troubleshooting

Common issues and solutions:

### Visual: Troubleshooting Guide

Diagnose and fix connection issues:

In [ ]:
# Troubleshooting Flowchart
diagram = """
flowchart TD
    Start["Connection Issue?"] --> Test["Run test_connection()"]
    
    Test --> Success{Connected?}
    
    Success -->|Yes| Working["✓ Ready to search"]
    
    Success -->|No| Error1{Error type?}
    
    Error1 -->|Authentication| Fix1["Check credentials in .env:<br/>ENDPOINT, KEY, INDEX"]
    
    Error1 -->|Index not found| Fix2["Verify index name<br/>matches Azure Portal"]
    
    Error1 -->|Network| Fix3["Check firewall rules<br/>and network access"]
    
    Error1 -->|Semantic config| Fix4["Verify Premium tier<br/>and semantic config"]
    
    Fix1 --> Retry["Retry connection"]
    Fix2 --> Retry
    Fix3 --> Retry
    Fix4 --> Retry
    
    Retry --> Test
    
    style Working fill:#4CAF50,color:#fff
    style Fix1 fill:#FF9800
    style Fix2 fill:#FF9800
    style Fix3 fill:#FF9800
    style Fix4 fill:#FF9800
"""
render_mermaid(diagram)

In [ ]:
# Test connection and index status
def test_connection():
    """Verify Azure Search connection and index configuration"""
    
    print("\n" + "="*80)
    print("Connection Diagnostics")
    print("="*80 + "\n")
    
    try:
        # Test simple search
        results = search_client.search(search_text="*", top=1)
        result_list = list(results)
        
        print("✅ Connection successful!")
        print(f"✅ Index '{index_name}' is accessible")
        print(f"✅ Document count: {len(result_list) if result_list else 'Unknown'}")
        
        # Check available fields
        if result_list:
            sample_doc = result_list[0]
            print(f"\n📋 Available fields in index:")
            for field in sample_doc.keys():
                if not field.startswith('@'):
                    print(f"   • {field}")
        
        return True
        
    except Exception as e:
        print(f"❌ Connection failed: {e}")
        print("\nTroubleshooting steps:")
        print("1. Verify AZURE_SEARCH_ENDPOINT in .env file")
        print("2. Verify AZURE_SEARCH_KEY (admin key required)")
        print("3. Verify AZURE_SEARCH_INDEX name is correct")
        print("4. Check if index exists in Azure Portal")
        return False

# Run diagnostics
test_connection()

## 10. Next Steps

Now that you've learned the three search types, consider:

1. **Optimize your index**
   - Configure semantic configuration for semantic search
   - Add vector fields for hybrid search
   - Set up scoring profiles for custom ranking

2. **Implement in your application**
   - Add search API endpoints
   - Implement pagination and filters
   - Add autocomplete for better UX

3. **Monitor and improve**
   - Track search analytics
   - Analyze query patterns
   - A/B test different search configurations

4. **Explore advanced features**
   - Knowledge Store integration
   - AI enrichment pipelines
   - Custom skills and analyzers

### Useful Resources:
- [Azure Search Documentation](https://docs.microsoft.com/azure/search/)
- [Python SDK Reference](https://docs.microsoft.com/python/api/azure-search-documents/)
- [Semantic Search Guide](https://docs.microsoft.com/azure/search/semantic-search-overview)

### Visual: Complete Search Flow

End-to-end query execution across all three methods:

---

## Summary

You've now learned:

✅ **Keyword Search** - Fast, precise, exact term matching  
✅ **Semantic Search** - AI-powered natural language understanding  
✅ **Hybrid Search** - Best of all methods combined

Choose the right search type based on your use case, budget, and quality requirements!

In [28]:
# Complete Search Flow
diagram = """
graph TB
    Start["Start Here"] --> Choose{Choose Search Type}
    
    Choose -->|Simple| KW["Keyword Search"]
    Choose -->|Smart| SEM["Semantic Search"]
    Choose -->|Best| HYB["Hybrid Search"]
    
    KW --> Query1["Execute search_client.search with search_text"]
    SEM --> Query2["Execute search_client.search with queryType='semantic'"]
    HYB --> Query3["Execute search_client.search with semantic + filters"]
    
    Query1 --> Index["Hotels Index in Azure"]
    Query2 --> Index
    Query3 --> Index
    
    Index --> Process1["BM25 Ranking"]
    Index --> Process2["L2 Semantic Model"]
    Index --> Process3["Hybrid Ranking"]
    
    Process1 --> Results1["Keyword Results"]
    Process2 --> Results2["Semantic Results + Captions"]
    Process3 --> Results3["Hybrid Results + Captions"]
    
    Results1 --> Display["Display in Notebook"]
    Results2 --> Display
    Results3 --> Display
    
    Display --> End["Analysis Complete"]
    
    style Start fill:#0078D4,color:#fff
    style Index fill:#4CAF50,color:#fff
    style End fill:#FF9800,color:#fff
"""
render_mermaid(diagram)